# 09 · Structured Credit & Tranches

Structured credit deals (CLO/ABS/RMBS) are described via JSON payloads containing pools, tranches, and waterfalls. This example reuses the helper module from the structured-credit capability script to build a schema-compliant spec and price it end-to-end.

### Learning Objectives
- Construct a minimal structured-credit JSON payload (assets, tranches, waterfall)
- Load it with `StructuredCredit.from_json` and price via the standard registry
- Request supported risk metrics (e.g., theta) and inspect the results

In [ ]:
import json
import runpy
from datetime import date
from pathlib import Path

from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import StructuredCredit
from finstack.valuations.pricer import standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (1.0, 0.9960), (3.0, 0.9820), (5.0, 0.9550), (10.0, 0.9050)],
    )
)
registry = standard_registry()

examples_dir = Path.cwd()
while examples_dir.name != "examples" and examples_dir != examples_dir.parent:
    examples_dir = examples_dir.parent

if examples_dir.name != "examples":
    raise FileNotFoundError(f"Could not locate the examples directory from {Path.cwd()}")

helpers_path = (
    examples_dir
    / "scripts"
    / "valuations"
    / "instruments"
    / "structured_credit_capabilities.py"
)
if not helpers_path.exists():
    raise FileNotFoundError(f"Structured credit helpers not found: {helpers_path}")
helpers_module = runpy.run_path(str(helpers_path))
base_deal_payload = helpers_module["base_deal_payload"]

deal_spec = base_deal_payload("CLO-DEMO", "CLO", asset_kind="loan")
spec_json = json.dumps(deal_spec)
structured_credit = StructuredCredit.from_json(spec_json)


## Price the Deal
The structured-credit pricer consumes the JSON spec, runs the waterfall, and returns PV plus requested metrics.

In [ ]:
metrics = ["theta"]
result = registry.price_with_metrics(
    structured_credit,
    "discounting",
    market,
    metrics,
    as_of=as_of,
)
print("PV:", result.value)
print("Theta (time decay):", result.measures.get("theta"))


## Summary
- Structured credit deals are provided as JSON documents referencing discount-curve IDs already loaded in `MarketContext`.
- Minimal specs include asset definitions, tranche stack, and a waterfall describing fee/interest/principal ordering.
- `StructuredCredit.from_json` plus `price_with_metrics` produces PV and supported risk metrics (e.g., theta); VaR/ES metrics require market history in `MarketContext`.